<div style='background:#000000;border-radius:10px;padding:26px 30px;border-top:6px solid #3777FF;border-bottom:3px solid #0EE071;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'>
<p style='color:#0EE071;margin:0;letter-spacing:2px;font-size:12px'>NOVA IMS · ENTERPRISE DATA SCIENCE BOOTCAMP 2026 · PROJETO D</p>
<h1 style='color:#FFFFFF;margin:6px 0 4px 0'>Telco Customer Churn — <span style='color:#3777FF'>Relatórios ydata-profiling</span></h1>
<p style='color:#FFFFFF;margin:0;font-size:15px'>Notebook 01b · Exploração automática em HTML, com a paleta do projeto</p>
</div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>Índice · O que este notebook gera</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Complementa o notebook <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>01_preprocessing_eda</code> com a abordagem sugerida na aula 6: relatórios HTML automáticos do <b style='color:#0EE071'>ydata-profiling</b>,
com a mesma paleta dos gráficos (<b style='color:#3777FF'>azul #3777FF</b>, <b style='color:#0EE071'>verde #0EE071</b>, preto).</p>
<ol style='margin:4px 0'>
<li><b style='color:#3777FF'>Base unida</b> — as 51 colunas, antes da catalogação (mostra constantes, duplicados e leakage nos <i>Alerts</i>)</li>
<li><b style='color:#3777FF'>Dataset final</b> — as 41 colunas de <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>train.csv</code>/<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>test.csv</code></li>
<li><b style='color:#0EE071'>Stayed vs Churned</b> — comparação lado a lado, variável a variável (<b style='color:#3777FF'>azul</b> vs <b style='color:#0EE071'>verde</b>)</li></ol>
<p>Os ficheiros ficam em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/profiling/</code> e abrem em qualquer browser. Cada relatório demora cerca de 30–60 segundos.</p></div></div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>1 · Setup</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Requer <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>ydata-profiling</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>setuptools&lt;81</code>: as versões recentes do setuptools removeram o módulo <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>pkg_resources</code>,
de que o ydata-profiling ainda depende. Ambos estão no <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>requirements.txt</code>. Python suportado: 3.10 a 3.13.</p>
<p>A primeira célula instala as dependências no kernel ativo, caso ainda não estejam instaladas.</p></div></div>

In [ ]:
import importlib.util, sys
if importlib.util.find_spec("ydata_profiling") is None:
    %pip install -q "ydata-profiling>=4.12" "setuptools<81"

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

from telco_churn import data, features, profiling
from telco_churn.config import TARGET

OPEN_IN_BROWSER = True   # abre cada relatório no browser assim que é gerado
SHOW_INLINE = False      # True para ver dentro do notebook (pesado no VS Code)

def show(report, path):
    print("Relatório:", path.relative_to(ROOT))
    if OPEN_IN_BROWSER:
        import webbrowser; webbrowser.open(path.as_uri())
    if SHOW_INLINE:
        report.to_notebook_iframe()

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>2 · Preparar os dados</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Reutiliza exatamente o mesmo código do notebook 01 (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>features</code>), garantindo que os relatórios descrevem
os mesmos dados que vão para o modelo.</p></div></div>

In [ ]:
df = data.merge_datasets(data.load_raw())
df_model, df_joined = features.split_population(df)
NUM_COLS, CAT_COLS, roles = features.feature_roles(df_model)   # antes das features criadas
df_model = features.add_features(df_model)

FEATURES = NUM_COLS + features.NEW_NUM + CAT_COLS + features.NEW_CAT
FINAL_COLS = ["Customer ID", "CLTV", "Churn Score", "Satisfaction Score"] + FEATURES + [TARGET]
print(f"Base unida: {df.shape} | Dataset final: {df_model[FINAL_COLS].shape} | Features: {len(FEATURES)}")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>3 · Relatório 1 — Base unida (51 colunas)</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Perfil da base completa, antes de qualquer exclusão. É útil para <b style='color:#0EE071'>justificar a catalogação</b>: os <i>Alerts</i> assinalam
colunas constantes (Country, State), alta cardinalidade (City, Lat Long), correlações perfeitas (Under 30 com Age, Total Revenue com Total Charges)
e os vazios de Churn Category/Reason.</p></div></div>

In [ ]:
rep_raw, path_raw = profiling.profile_dataset(
    df, title="Telco Churn — Base unida (51 colunas)", file_name="01_base_unida",
    description="União dos 5 ficheiros por Customer ID e Zip Code: 7.043 clientes, antes da catalogação.")
show(rep_raw, path_raw)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>4 · Relatório 2 — Dataset final (41 colunas)</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>As colunas que seguem para a modelação, com uma descrição curta em cada variável.
Confirmar: <b style='color:#0EE071'>0 missing</b>, <b style='color:#0EE071'>0 duplicados</b>, distribuições e correlações das features criadas.</p></div></div>

In [ ]:
rep_final, path_final = profiling.profile_dataset(
    df_model[FINAL_COLS], title="Telco Churn — Dataset final", file_name="02_dataset_final",
    description="População de modelação (Churned + Stayed): 6.589 clientes, 41 colunas.")
show(rep_final, path_final)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>5 · Relatório 3 — Stayed vs Churned</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O relatório mais útil para o problema: cada variável aparece com as estatísticas e o histograma dos dois grupos sobrepostos,
<b style='color:#3777FF'>Stayed a azul</b> e <b style='color:#0EE071'>Churned a verde</b>. Diferenças grandes indicam variáveis com poder discriminante.</p>
<p>Exemplos a procurar: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Tenure in Months</code> (churners com antiguidade muito menor), <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Contract</code> (Month-to-Month domina nos churners),
<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Number of Referrals</code> (94% de zeros nos churners vs 70% nos que ficam).</p></div></div>

In [ ]:
rep_cmp, path_cmp = profiling.profile_by_target(df_model, TARGET, FEATURES, file_name="03_stayed_vs_churned")
show(rep_cmp, path_cmp)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>6 · Como ler e o que levar para o relatório</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>Alerts</b> — ponto de partida: confirmar que cada alerta está tratado na catalogação (removido, mantido com justificação ou testado)</li><li><b style='color:#0EE071'>Variables</b> — distribuição, zeros, extremos e frequências de cada coluna</li><li><b style='color:#0EE071'>Correlations</b> — Spearman e Cramér's V na escala azul → branco → verde</li><li><b style='color:#0EE071'>Interactions</b> — dispersão de cada variável contra Tenure, Monthly Charge e churn</li><li><b style='color:#0EE071'>Missing values</b> — confirma que não há missing no dataset final</li></ul>
<p>Os relatórios complementam, não substituem, os gráficos do notebook 01: aqueles mostram a <b style='color:#3777FF'>taxa de churn por categoria</b>
e o <b style='color:#3777FF'>Cramér's V</b> com o target, que o ydata-profiling não calcula diretamente.</p></div></div>